In [3]:
import pandas as pd 
import nltk

In [ ]:
# get the data from spam file and get it a data frame

message = pd.read_csv("SMSSpamCollection.txt", # get the data 
                      sep="\t", # as data is tab separated
                      names= ["label","message"]
                      )  # name the columns

In [5]:
message 

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


Prepare the Target values 

In [6]:
#Make the Target data 
#Perform Onehot encoding on the ham/spam binary ouput colum 
#For spam or not True and false is better than 0/1
y = pd.get_dummies(message["label"]) #Convert the binary column to 1/0
y = y.iloc[:,1].values #merge back the columns 

Text PreProcessing  : 


Do this before breaking the target , input type so that everything is clean and then we make the test train split and move forward but only apply text preprocessing to the messages themselves

Tokenization , stopwords , stemming,lemmatization,NLTK

In [7]:
#Proper Text tokenization 
#  Get the sentence remove clutter -> lower case the sentence  -> Break the words based on spaces-> 
import re
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

from nltk.stem import WordNetLemmatizer # For lemmatizing
ps = WordNetLemmatizer() #Lemmatize object

corpus = []

for i in range(len(message)): # As messages have 2 colums and both needs to be checked
    #Remove all special character
    changed = re.sub("[^a-zA-Z0-9]"," ",message["message"][i])
    #Lowecase All
    changed = changed.lower()
    #Split the sentences into array form based on spaces
    changed = changed.split()
    #apply Lematize to all the words
    changed = [ps.lemmatize(word) for word in changed if not word in stop_words]
    #Combine the words back into sentences
    changed = " ".join(changed)
    corpus.append(changed)


In [8]:
corpus

['go jurong point crazy available bugis n great world la e buffet cine got amore wat',
 'ok lar joking wif u oni',
 'free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry question std txt rate c apply 08452810075over18',
 'u dun say early hor u c already say',
 'nah think go usf life around though',
 'freemsg hey darling 3 week word back like fun still tb ok xxx std chgs send 1 50 rcv',
 'even brother like speak treat like aid patent',
 'per request melle melle oru minnaminunginte nurungu vettam set callertune caller press 9 copy friend callertune',
 'winner valued network customer selected receivea 900 prize reward claim call 09061701461 claim code kl341 valid 12 hour',
 'mobile 11 month u r entitled update latest colour mobile camera free call mobile update co free 08002986030',
 'gonna home soon want talk stuff anymore tonight k cried enough today',
 'six chance win cash 100 20 000 pound txt csh11 send 87575 cost 150p day 6days 16 tsandcs apply reply

Text PreProcessing -2 

1. Apply Bag Of words

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=2500,binary=True,ngram_range=(1,3))

In [ ]:
#Make the input data 
X = cv.fit_transform(corpus).toarray() 
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5572, 2500))

In [11]:
y

array([False, False,  True, ..., False, False, False], shape=(5572,))

Now Prepare the TrainTest Split 

Start Model Preparation


In [ ]:
from sklearn.naive_bayes import MultinomialNB
spam_detector = MultinomialNB().fit(x_train,y_train)

In [15]:
y_pred = spam_detector.predict(x_test)

Get the accuracy of the ml model

In [16]:
from sklearn.metrics import accuracy_score,classification_report

acc_score = accuracy_score(y_test,y_pred)
class_score= classification_report(y_test,y_pred)

print(f"Model Accuracy : {acc_score}")
print(f"Model Classification Report :\n {class_score}")


Model Accuracy : 0.9857519788918205
Model Classification Report :
               precision    recall  f1-score   support

       False       0.99      1.00      0.99      1628
        True       0.98      0.92      0.95       267

    accuracy                           0.99      1895
   macro avg       0.98      0.96      0.97      1895
weighted avg       0.99      0.99      0.99      1895



2. Using TF-IDF

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [18]:
tfidf = TfidfVectorizer(max_features=2500,ngram_range = (1,3))

In [19]:
#Only need to Update the labels as they remain the same true and flase but the messages themselves needs to be updated
X = tfidf.fit_transform(corpus).toarray()
cv.vocabulary_

{'go': np.int64(928),
 'point': np.int64(1688),
 'crazy': np.int64(606),
 'available': np.int64(284),
 'bugis': np.int64(386),
 'great': np.int64(975),
 'world': np.int64(2447),
 'la': np.int64(1186),
 'cine': np.int64(501),
 'got': np.int64(966),
 'wat': np.int64(2359),
 'ok': np.int64(1556),
 'lar': np.int64(1197),
 'joking': np.int64(1156),
 'wif': np.int64(2411),
 'free': np.int64(859),
 'entry': np.int64(758),
 'wkly': np.int64(2435),
 'comp': np.int64(548),
 'win': np.int64(2417),
 'cup': np.int64(616),
 'final': np.int64(825),
 'may': np.int64(1374),
 'text': np.int64(2112),
 'receive': np.int64(1763),
 'question': np.int64(1738),
 'std': np.int64(2015),
 'txt': np.int64(2218),
 'rate': np.int64(1746),
 'apply': np.int64(249),
 'free entry': np.int64(867),
 'txt rate': np.int64(2225),
 'rate apply': np.int64(1747),
 'txt rate apply': np.int64(2226),
 'dun': np.int64(718),
 'say': np.int64(1837),
 'early': np.int64(725),
 'already': np.int64(225),
 'nah': np.int64(1486),
 'think'

Make the Train Test Split

In [23]:
from sklearn.model_selection import train_test_split as tts
x_train,x_test,y_train,y_test = tts(X,y,test_size=0.34,random_state=23)

In [24]:
#Recreate the mode
from sklearn.naive_bayes import MultinomialNB
spam_detector = MultinomialNB().fit(x_train,y_train)

y_pred = spam_detector.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score,classification_report

acc_score = accuracy_score(y_test,y_pred)
class_score= classification_report(y_test,y_pred)

print(f"Model Accuracy : {acc_score}")
print(f"Model Classification Report :\n {class_score}")


Model Accuracy : 0.9794195250659631
Model Classification Report :
               precision    recall  f1-score   support

       False       0.98      1.00      0.99      1628
        True       1.00      0.85      0.92       267

    accuracy                           0.98      1895
   macro avg       0.99      0.93      0.95      1895
weighted avg       0.98      0.98      0.98      1895



This Time on Random Forest

In [27]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier()
classifier.fit(x_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [28]:
y_pred = classifier.predict(x_test)

In [29]:
acc_score = accuracy_score(y_test,y_pred)
class_score= classification_report(y_test,y_pred)

print(f"Model Accuracy : {acc_score}")
print(f"Model Classification Report :\n {class_score}")

Model Accuracy : 0.9815303430079155
Model Classification Report :
               precision    recall  f1-score   support

       False       0.98      1.00      0.99      1628
        True       0.99      0.88      0.93       267

    accuracy                           0.98      1895
   macro avg       0.98      0.94      0.96      1895
weighted avg       0.98      0.98      0.98      1895

